# 05 — SHAP Diagnostic Study

**Purpose:** diagnose *why* subject-independent accuracy drops relative to
subject-dependent accuracy, rather than chase a new state of the art.

We train four comparators on the flattened DE feature vector (`14 x 5 = 70` dims):
**KNN, Random Forest, LDA, and a small CNN** (`src/models/shap_diagnostic.py`).
After training we compute **SHAP values** for the tree model and the CNN and look
at how much the feature attributions vary *across subjects* — large cross-subject
variance in the important features is the mechanism behind the generalization gap.

## 1. Setup

In [ ]:
import os, sys, time, json
from functools import partial

import numpy as np
import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.evaluation import (
    set_seed, get_device, count_parameters,
    make_default_fit, default_predict,
    run_subject_dependent, run_loso, save_results, quick_report,
)
from src.data_loader import (
    build_subject_dependent_loaders, build_loso_loaders, load_subject, REDUCED_5_CHANNELS,
)

set_seed(42)
DEVICE = get_device()

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
SD_SPLITS     = os.path.join(ROOT, "data", "splits", "subject_dependent_splits.json")
LOSO_SPLITS   = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR   = os.path.join(ROOT, "results")

# ----- experiment configuration -----
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> a 3-subject / 3-fold smoke test with few epochs
                      # False -> the full benchmark used for the thesis table
print("device:", DEVICE, "| label:", LABEL, "| QUICK:", QUICK)

import matplotlib.pyplot as plt
import shap
from sklearn.metrics import accuracy_score, f1_score

from src.evaluation import compute_metrics, summarize_per_fold, save_results
from src.models import shap_diagnostic
from src.models.shap_diagnostic import (
    make_sklearn_models, SimpleDECNN, loader_to_xy, torch_predict_proba,
    subject_shap_variance, DE_FEATURE_NAMES,
)
MODEL_NAME = "SHAP_CNN"          # the CNN row that also feeds the aggregation table
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

## 2. Helpers

`fetch_xy(...)` pulls a `(X[N,70], y, subject_ids)` triple for any subject list,
reusing the shared normalizer-aware loaders so the DE features are scaled exactly
as the other notebooks see them.

In [ ]:
def _pick_positive(sv):
    """Return the positive-class SHAP matrix [N, F] across shap versions."""
    if isinstance(sv, list):
        return np.asarray(sv[1] if len(sv) > 1 else sv[0])
    sv = np.asarray(sv)
    if sv.ndim == 3:            # [N, F, n_classes]
        return sv[:, :, -1]
    return sv                   # already [N, F]

def _cnn_fit(model, Xtr, ytr, epochs, lr=1e-3, bs=64):
    model.to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    ce = torch.nn.CrossEntropyLoss()
    Xt = torch.tensor(Xtr.reshape(-1, 14, 5), dtype=torch.float32)
    yt = torch.tensor(ytr, dtype=torch.int64)
    ds = torch.utils.data.TensorDataset(Xt, yt)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=True)
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = ce(model(xb), yb); loss.backward(); opt.step()
    return model

def _cnn_predict(model, X):
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X.reshape(-1, 14, 5), dtype=torch.float32, device=DEVICE)
        return model(t).argmax(1).cpu().numpy()

## 3. Subject-dependent protocol — all four models

For every subject we read its 80/20 split, fit each model, and record accuracy /
macro-F1. The CNN's per-subject predictions are pooled to a single `SHAP_CNN`
subject-dependent result in the shared schema.

In [ ]:
with open(SD_SPLITS) as f:
    sd_spec = json.load(f)["subjects"]
sd_subjects = list(sd_spec)[:3] if QUICK else list(sd_spec)
EPOCHS = 15 if QUICK else 40

sd_rows = []            # (model, subject, acc, f1)
cnn_sd_perfold = []     # schema dicts for the CNN

for sid in sd_subjects:
    tl, vl, _ = build_subject_dependent_loaders(
        PROCESSED_DIR, SD_SPLITS, sid, fold=0, label=LABEL, feature="de",
        batch_size=256, normalize=True,
    )
    Xtr, ytr, _ = loader_to_xy(tl)
    Xte, yte, _ = loader_to_xy(vl)

    for mname, pipe in make_sklearn_models().items():
        pipe.fit(Xtr, ytr); pred = pipe.predict(Xte)
        sd_rows.append((mname, sid, accuracy_score(yte, pred), f1_score(yte, pred, average="macro")))

    cnn = _cnn_fit(SimpleDECNN(n_classes=2), Xtr, ytr, EPOCHS)
    pred = _cnn_predict(cnn, Xte)
    sd_rows.append(("CNN", sid, accuracy_score(yte, pred), f1_score(yte, pred, average="macro")))
    m = compute_metrics(yte, pred); m["subject"] = sid
    cnn_sd_perfold.append(m)

import pandas as pd
sd_df = pd.DataFrame(sd_rows, columns=["model", "subject", "accuracy", "f1"])
print(sd_df.groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]).round(4))

cnn_params = count_parameters(SimpleDECNN(n_classes=2))
sd_res = summarize_per_fold(cnn_sd_perfold, cnn_params, train_time_sec=float("nan"),
                            protocol="subject_dependent")

## 4. Subject-independent (LOSO) protocol — all four models

In [ ]:
with open(LOSO_SPLITS) as f:
    loso = json.load(f)
folds = list(range(3)) if QUICK else list(range(loso["n_folds"]))

loso_rows = []
cnn_loso_perfold = []
# keep the pooled LOSO test set + subject ids for the SHAP cross-subject analysis
shap_X, shap_y, shap_sid = [], [], []
last_cnn = None

for k in folds:
    tl, vl, el, meta = build_loso_loaders(
        PROCESSED_DIR, LOSO_SPLITS, fold=k, label=LABEL, feature="de",
        batch_size=512, normalize=True, val_frac=0.0,
    )
    Xtr, ytr, _ = loader_to_xy(tl)
    Xte, yte, sid_te = loader_to_xy(el)

    for mname, pipe in make_sklearn_models().items():
        pipe.fit(Xtr, ytr); pred = pipe.predict(Xte)
        loso_rows.append((mname, meta["test_subject"],
                          accuracy_score(yte, pred), f1_score(yte, pred, average="macro")))

    cnn = _cnn_fit(SimpleDECNN(n_classes=2), Xtr, ytr, EPOCHS)
    last_cnn = cnn
    pred = _cnn_predict(cnn, Xte)
    loso_rows.append(("CNN", meta["test_subject"],
                      accuracy_score(yte, pred), f1_score(yte, pred, average="macro")))
    m = compute_metrics(yte, pred); m["fold"] = k; m["test_subject"] = meta["test_subject"]
    cnn_loso_perfold.append(m)

    # subsample for SHAP to keep KernelExplainer tractable
    take = np.random.RandomState(0).choice(len(Xte), size=min(60, len(Xte)), replace=False)
    shap_X.append(Xte[take]); shap_y.append(yte[take]); shap_sid.append(sid_te[take])

loso_df = pd.DataFrame(loso_rows, columns=["model", "subject", "accuracy", "f1"])
print(loso_df.groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]).round(4))

loso_res = summarize_per_fold(cnn_loso_perfold, cnn_params, train_time_sec=float("nan"),
                              protocol="loso")
save_results(MODEL_NAME, subject_dependent=sd_res, loso=loso_res,
             family="Diagnostic (DE + classical/CNN)", results_dir=RESULTS_DIR,
             config={"label": LABEL, "feature": "de_flat_70", "quick": QUICK})

# also stash the full four-model comparison for the appendix
sd_df.assign(protocol="subject_dependent").to_csv(
    os.path.join(RESULTS_DIR, "SHAP_diagnostic_models_sd.csv"), index=False)
loso_df.assign(protocol="loso").to_csv(
    os.path.join(RESULTS_DIR, "SHAP_diagnostic_models_loso.csv"), index=False)

## 5. SHAP — Random Forest (TreeExplainer, LOSO)

The Random Forest is refit on the union of all-but-one LOSO fold's training data
so we have one global model to explain; `TreeExplainer` is exact and fast.

In [ ]:
X_all = np.concatenate(shap_X); y_all = np.concatenate(shap_y); sid_all = np.concatenate(shap_sid)

rf = make_sklearn_models()["RandomForest"]
# fit on a broad slice (first fold's training set is representative enough here)
tl0, _, _, _ = build_loso_loaders(PROCESSED_DIR, LOSO_SPLITS, fold=folds[0], label=LABEL,
                                  feature="de", batch_size=512, normalize=True)
Xfit, yfit, _ = loader_to_xy(tl0)
rf.fit(Xfit, yfit)

explainer = shap.TreeExplainer(rf.named_steps["clf"])
Xs = rf.named_steps["scale"].transform(X_all)
sv_pos = _pick_positive(explainer.shap_values(Xs))    # [N, 70]

plt.figure()
shap.summary_plot(sv_pos, Xs, feature_names=DE_FEATURE_NAMES, show=False, max_display=15)
plt.title("SHAP summary — Random Forest (LOSO pooled test)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "shap_summary_rf_loso.png"), dpi=150)
plt.show()

## 6. SHAP — small CNN (KernelExplainer, LOSO)

`KernelExplainer` is model-agnostic; we give it a 50-row background sample and
explain the same pooled test rows.

In [ ]:
bg = shap.sample(X_all, 50, random_state=0)
f = torch_predict_proba(last_cnn, DEVICE)
kexp = shap.KernelExplainer(f, bg)
# ASSUMPTION: nsamples kept small (128) for runtime; raise for smoother estimates.
sv_cnn_pos = _pick_positive(kexp.shap_values(X_all[:120], nsamples=128))

plt.figure()
shap.summary_plot(sv_cnn_pos, X_all[:120], feature_names=DE_FEATURE_NAMES,
                  show=False, max_display=15)
plt.title("SHAP summary — small CNN (LOSO pooled test)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "shap_summary_cnn_loso.png"), dpi=150)
plt.show()

## 7. Cross-subject SHAP variance — the diagnostic figure

For each feature we take the mean `|SHAP|` *within* each left-out subject, then the
**variance of that quantity across subjects**. Features with high cross-subject
variance are the ones whose predictive role is subject-specific — i.e. what a
subject-independent model cannot rely on. This plot is the crux of the chapter.

In [ ]:
var_rf = subject_shap_variance(sv_pos, sid_all)
order = np.argsort(var_rf)[::-1][:15]

plt.figure(figsize=(7, 5))
plt.barh([DE_FEATURE_NAMES[i] for i in order][::-1], var_rf[order][::-1])
plt.xlabel("variance across subjects of mean |SHAP|")
plt.title("Subject-specificity of DE features (Random Forest)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "shap_subject_variance_rf.png"), dpi=150)
plt.show()

pd.DataFrame({"feature": DE_FEATURE_NAMES, "cross_subject_shap_variance": var_rf}) \
    .sort_values("cross_subject_shap_variance", ascending=False) \
    .to_csv(os.path.join(RESULTS_DIR, "shap_subject_variance_rf.csv"), index=False)

## Notes for the methodology chapter

* All four comparators use the identical 70-D DE vector and the shared
  normalizer-aware loaders — differences are purely model-driven.
* `SHAP_CNN_results.json` feeds the consolidated table in notebook 09; the full
  KNN/RF/LDA/CNN comparison lives in `results/SHAP_diagnostic_models_*.csv`.
* SHAP sample sizes (`nsamples`, background size, pooled-row subsample) are set
  for runtime, not precision — see the `# ASSUMPTION` comments.
* `train_time_sec` is recorded as `NaN` here because this notebook interleaves
  four models per fold; efficiency claims should use the dedicated notebooks.